In [1]:
import os

In [4]:
pwd

'/Users/basazinbelhu/Simple_MLOPS'

In [5]:
# os.chdir("../")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [7]:
from src.simple_mlops.constants import *
from src.simple_mlops.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifact_dir]) 
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])          # <- list brackets, like your other calls

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )
        return data_transformation_config

In [14]:
import os
from src.simple_mlops import logger
from sklearn.model_selection import train_test_split
import pandas as pd


In [15]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

from src.simple_mlops import logger
# from src.simple_mlops.entity.config_entity import DataTransformationConfig


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def train_test_splitting(self):
        data = pd.read_csv(self.config.data_path, sep=";")   # semicolon-delimited!

        train, test = train_test_split(data, test_size=0.25, random_state=42)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Split data into training and test sets")
        logger.info(f"Train shape: {train.shape}")
        logger.info(f"Test shape: {test.shape}")

In [16]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_splitting()
except Exception as e:
    raise e 

[2026-06-12 22:27:57,131: INFO: common]: Successfully read YAML file: config/config.yaml
[2026-06-12 22:27:57,132: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-12 22:27:57,135: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-12 22:27:57,135: INFO: common]: Directory created or already exists: artifact
[2026-06-12 22:27:57,137: INFO: common]: Directory created or already exists: artifact/data_transformation
[2026-06-12 22:27:57,173: INFO: 1508149950]: Split data into training and test sets
[2026-06-12 22:27:57,174: INFO: 1508149950]: Train shape: (1199, 12)
[2026-06-12 22:27:57,174: INFO: 1508149950]: Test shape: (400, 12)
